In [12]:
from pathlib import Path
import matplotlib as mpl
from omegaconf import OmegaConf
from torch import Tensor
import torch
from train import Trainer
from src.interfaces import DatasetBase
from src.datasets.addition import AdditionDataset

mpl.rcParams['figure.dpi'] = 300

### Utils

In [6]:
def pretty_print_sample(dataset: DatasetBase, x: Tensor, verbose: bool = True) -> str:
    steps = [
        dataset.to_string(x[i]).split("\n")
        for i in range(x.shape[0])
    ]

    x_str = "\n".join(
        "   ".join(
            steps[i][j]
            for i in range(len(steps))
        )
        for j in range(len(steps[0]))
    )

    if verbose:
        print(x_str)

    return x_str

### Checkpoint options

In [34]:
ckpt = "ckpt_15.pth"
output_dir = Path.cwd().parent / "outputs/multi/2025-12-12/17-01-46"
config_file = output_dir / ".hydra" / "config.yaml"
cfg_multi = OmegaConf.load(config_file)

cfg_multi.resume_ckpt = output_dir / "checkpoints" / ckpt

In [14]:
ckpt = "ckpt_13.pth"
output_dir = Path.cwd().parent / "outputs/single/2025-12-12/16-41-08"
config_file = output_dir / ".hydra" / "config.yaml"
cfg_single = OmegaConf.load(config_file)

cfg_single.resume_ckpt = output_dir / "checkpoints" / ckpt
cfg_single.dataset.max_digits = 4

### Load Model

In [35]:
trainer_multi = Trainer(cfg_multi, output_dir)
trainer_multi.model.eval()

/mnt/g/Projects/Deep-Learning-2025/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


_FabricModule(
  (_forward_module): Transformer(
    (src_embed): Embedding(16, 128)
    (tgt_embed): Embedding(16, 128)
    (pos_encoding): IdentityEncoding()
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x EncoderLayer(
          (mha): MultiHeadAttention(
            (query): Linear(in_features=128, out_features=128, bias=True)
            (key): Linear(in_features=128, out_features=128, bias=True)
            (value): Linear(in_features=128, out_features=128, bias=True)
            (output): Linear(in_features=128, out_features=128, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
            (relative_pos_encoding): RotaryPositionalEncoding(
              (rope): RotaryEmbedding()
            )
          )
          (mha_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (ff): FeedForward(
            (ff): Sequential(
              (0): GatedLU(
                (proj): Linear(in_features=128, out_features=512, b

In [15]:
trainer_single = Trainer(cfg_single, output_dir)
trainer_single.model.eval()

/mnt/g/Projects/Deep-Learning-2025/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


_FabricModule(
  (_forward_module): Transformer(
    (src_embed): Embedding(16, 128)
    (tgt_embed): Embedding(16, 128)
    (pos_encoding): IdentityEncoding()
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x EncoderLayer(
          (mha): MultiHeadAttention(
            (query): Linear(in_features=128, out_features=128, bias=True)
            (key): Linear(in_features=128, out_features=128, bias=True)
            (value): Linear(in_features=128, out_features=128, bias=True)
            (output): Linear(in_features=128, out_features=128, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
            (relative_pos_encoding): RotaryPositionalEncoding(
              (rope): RotaryEmbedding()
            )
          )
          (mha_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (ff): FeedForward(
            (ff): Sequential(
              (0): GatedLU(
                (proj): Linear(in_features=128, out_features=512, b

### Analysis

#### Get Sample

In [ ]:
dataset = AdditionDataset(max_digits=4)

In [44]:
sample = dataset.get_example().to(trainer_multi.model.device)
sample_str = pretty_print_sample(dataset, sample)

  ________     ________     ______1_     ______1_     _____11_     _____11_     ____111_     ____111_     ___0111_     ___0111_     __00111_     __00111_     _100111_     _100111_     0100111_     0100111_
   8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542
+  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698
----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------
  ________     _______0     _______0     ______40     ______40     _____240     _____240     ____9240     ____9240     ___89240     ___89240     __289240     __289240     _9289

#### Run Model

In [45]:
x0 = sample[0:1]
x0_str = dataset.to_string(x0[0])

steps = [x0.clone()]
enc_attnss = []
y_preds = []

total_accuracy = 0.0
for i in range(sample.shape[0]-1):
  with torch.no_grad():
      _, accuracy, y_pred, _, _, enc_attns, _, _ = trainer_single.forward((x0, sample[i+1:i+2]))
      out = trainer_single.head.step(x0, y_pred)

      steps.append(out)
      y_preds.append(y_pred)
      enc_attnss.append(enc_attns)
      x0 = out
      total_accuracy += accuracy.item()

total_accuracy /= (sample.shape[0]-1)

out_str = pretty_print_sample(dataset, torch.cat(steps, dim=0), verbose=False)

print(f"=== Solution ===\n{sample_str}")
print(f"=== Prediction ===\n{out_str}")
print(f"Accuracy: {100*total_accuracy:.2f}%")

=== Solution ===
  ________     ________     ______1_     ______1_     _____11_     _____11_     ____111_     ____111_     ___0111_     ___0111_     __00111_     __00111_     _100111_     _100111_     0100111_     0100111_
   8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542
+  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698
----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------
  ________     _______0     _______0     ______40     ______40     _____240     _____240     ____9240     ____9240     ___89240     ___89240     __289240     _

In [46]:
x0 = sample[0:1]
x0_str = dataset.to_string(x0[0])

steps = [x0.clone()]
enc_attnss = []
y_preds = []
total_accuracy = 0.0

for _ in range(sample.shape[0]-1):
  with torch.no_grad():
      _, accuracy, y_pred, _, _, enc_attns, _, _ = trainer_multi.forward((x0, sample[i+1:i+2]))
      out = trainer_multi.head.step(x0, y_pred)

      steps.append(out)
      y_preds.append(y_pred)
      enc_attnss.append(enc_attns)
      x0 = out
      total_accuracy += accuracy.item()
      
total_accuracy /= sample.shape[0]-1
out_str = pretty_print_sample(dataset, torch.cat(steps, dim=0), verbose=False)

print(f"=== Solution ===\n{sample_str}")
print(f"=== Prediction ===\n{out_str}")
print(f"Accuracy: {total_accuracy}")

=== Solution ===
  ________     ________     ______1_     ______1_     _____11_     _____11_     ____111_     ____111_     ___0111_     ___0111_     __00111_     __00111_     _100111_     _100111_     0100111_     0100111_
   8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542      8637542
+  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698   +  0651698
----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------   ----------
  ________     _______0     _______0     ______40     ______40     _____240     _____240     ____9240     ____9240     ___89240     ___89240     __289240     _